In [ ]:
%pip install pandas

In [ ]:
import pandas as pd        
import datetime as dt       

# Read the hotel booking CSV file and store it as a table called df_otb
# df = dataframe (a table), otb = on the books
df_otb = pd.read_csv('/Users/josephtran/Documents/hotel_forecast/PEB - BI Project Files/data-otb.csv')

# Read the events CSV file and store it as a table called df_events
df_events = pd.read_csv('/Users/josephtran/Documents/hotel_forecast/PEB - BI Project Files/data-events.csv')

# Right now Python sees dates as plain text like "2025-01-06"
# Converting them to datetime objects lets Python do math on dates

# Convert the snapshot date column — the date the data was pulled
df_otb['snapshot_date'] = pd.to_datetime(df_otb['snapshot_date'])

# Convert the business date column — the actual night a guest is staying
df_otb['business_date'] = pd.to_datetime(df_otb['business_date'])

# Convert the event date column in the events table
df_events['event_date'] = pd.to_datetime(df_events['event_date'])

# Remove any cancelled events so they don't inflate our demand signals
# We only want events that are actually happening
df_events = df_events[df_events['is_cancelled'] == False]

# Calculate total room capacity for each hotel
# Total rooms = rooms sold / occupancy rate
# Example: 79 rooms sold at 25.08% occupancy = 315 total rooms
df_otb['total_rooms'] = round(df_otb['rooms_sold'] / df_otb['occupancy'])

# Calculate lead time — how many days in advance this snapshot was taken before the guest arrives
# Example: snapshot on June 1st for a July 4th stay = 33 days lead time
df_otb['lead_time_days'] = (df_otb['business_date'] - df_otb['snapshot_date']).dt.days

# Calculate what day of the week each business date falls on(weekdays vs weekends)
# 0 = Monday, 1 = Tuesday, 2 = Wednesday, 3 = Thursday, 4 = Friday, 5 = Saturday, 6 = Sunday
df_otb['day_of_week'] = df_otb['business_date'].dt.dayofweek

print("Phase 1 Complete: OTB Data Loaded Successfully!")

df_otb.head()

In [ ]:
# The events file has one row per event — a single date could have 10 events
# So we group them first to get one clean row per city per date
daily_events = df_events.groupby(['market', 'event_date']).agg(
    total_event_visitors=('visitors', 'sum'),  # Add up all visitor counts for that city/date
    event_count=('event_id', 'count')          # Count how many events are happening that day
).reset_index()                                # Turn the groupby result back into a normal table

# Now we connect the hotel data with the events summary
# 'how=left' means: keep every hotel row even if there are no events that day
df_combined = pd.merge(
    df_otb,           # Start with all the hotel booking rows
    daily_events,     # Bring in the events summary we just built
    how='left',       # Keep all hotel rows — don't drop dates with no events
    left_on=['location', 'business_date'],   # Hotel side: match by city and date
    right_on=['market', 'event_date']        # Events side: match by market and event date
)

# After the merge, dates with no events will show up as blank (NaN) - zero events
df_combined['total_event_visitors'] = df_combined['total_event_visitors'].fillna(0) 
df_combined['event_count'] = df_combined['event_count'].fillna(0)                   

# 'market' and 'event_date' are duplicates of 'location' and 'business_date'
# Dropping them keeps the table clean and easy to read
df_combined = df_combined.drop(columns=['market', 'event_date'])

print("Phase 2 Complete: Data Merged Successfully!")
df_combined.head()  # Show the first 5 rows to confirm everything looks right

In [ ]:
import numpy as np

# Find the most recent date the data was pulled — this is our "today"
latest_snapshot = df_combined['snapshot_date'].max()

# Keep only rows from the latest snapshot — this gives us the freshest picture of bookings
df_forecast = df_combined[df_combined['snapshot_date'] == latest_snapshot].copy()

# Keep only future dates within the next 90 days — that's our forecast window
df_forecast = df_forecast[
    (df_forecast['business_date'] > latest_snapshot) &                          # Must be after today
    (df_forecast['business_date'] <= latest_snapshot + pd.Timedelta(days=90))  # Must be within 90 days
]

# Start every date with a baseline assumption:
# We expect to fill 20% of the rooms that are still empty
df_forecast['expected_pickup_pct'] = 0.20

# Rule 1 — Weekend boost
# Fridays and Saturdays naturally get more bookings than weekdays
# So we add 10% more expected pickup on those days
# (day_of_week: 4 = Friday, 5 = Saturday)
df_forecast.loc[df_forecast['day_of_week'].isin([4, 5]), 'expected_pickup_pct'] += 0.10

# Rule 2 — Small event boost
# If a date has more than 1,000 visitors coming to the area, demand goes up
# We add 20% more expected pickup to account for that
df_forecast.loc[df_forecast['total_event_visitors'] > 1000, 'expected_pickup_pct'] += 0.20

# Rule 3 — Massive event boost
# If a date has more than 10,000 visitors (think Harborfest, Abbot Kinney)
# demand is so high we add another 40% on top
# Note: this stacks with Rule 2, so huge event dates get +60% total boost
df_forecast.loc[df_forecast['total_event_visitors'] > 10000, 'expected_pickup_pct'] += 0.40

# Forecasted pickup = how many of the remaining empty rooms we expect to sell
# Formula: empty rooms left × pickup percentage we calculated above
df_forecast['forecasted_pickup'] = round(df_forecast['left_to_sell'] * df_forecast['expected_pickup_pct'])

# Final forecasted rooms = rooms already booked + the new rooms we expect to sell
df_forecast['forecasted_rooms'] = df_forecast['rooms_sold'] + df_forecast['forecasted_pickup']

# Safety cap — we can never sell more rooms than the hotel physically has
# If the forecast goes over total capacity, we bring it back down to 100%
df_forecast['forecasted_rooms'] = np.minimum(df_forecast['forecasted_rooms'], df_forecast['total_rooms'])

# Convert forecasted rooms into an occupancy percentage
# Formula: forecasted rooms sold / total rooms in the hotel
df_forecast['forecast_occupancy'] = df_forecast['forecasted_rooms'] / df_forecast['total_rooms']

# Convert forecasted rooms into a revenue number
# Formula: forecasted rooms sold × average daily rate (price per room)
df_forecast['forecast_revenue'] = df_forecast['forecasted_rooms'] * df_forecast['adr_sold']

df_clean = df_forecast[['location', 'business_date', 'forecasted_pickup',
                         'forecasted_rooms', 'forecast_revenue', 'forecast_occupancy']].copy()

# Convert occupancy from decimal to percentage — 0.619 becomes 61.9%
df_clean['forecast_occupancy'] = (df_clean['forecast_occupancy'] * 100).round(1).astype(str) + '%'

# Format revenue as dollar amount with commas — 40573.65 becomes $40,573.65
df_clean['forecast_revenue'] = df_clean['forecast_revenue'].round(2).apply(lambda x: f'${x:,.2f}')

# Convert pickup and rooms to clean whole numbers — 30.0 becomes 30
df_clean['forecasted_pickup'] = df_clean['forecasted_pickup'].astype(int)
df_clean['forecasted_rooms'] = df_clean['forecasted_rooms'].astype(int)

df_clean.to_csv('90_day_forecast_clean.csv', index=False)

# Print confirmation messages first
print("Phase 3 Complete: 90-Day Forecast Generated!")
print("Clean forecast exported to 90_day_forecast_clean.csv — open in Excel or Google Sheets to view all 180 rows")
print()

# Show preview last so it renders as a formatted table — must be the final line
df_clean.head(10)

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter, FuncFormatter

df_boston = df_forecast[df_forecast['location'] == 'Boston, MA'].copy()

fig, ax1 = plt.subplots(figsize=(15, 7), facecolor='white')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()

events_bars = ax2.bar(
    df_boston['business_date'],
    df_boston['total_event_visitors'],
    alpha=0.15,
    color='#800080',
    width=1.2,
    label='Event Visitors'
)

ax2.set_ylabel('Total Event Visitors', color='#800080', fontsize=12, fontweight='bold', labelpad=15)
ax2.grid(False)

def thousands_formatter(x, pos):
    return f'{int(x/1000)}K' if x >= 1000 else str(int(x))
ax2.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))

ax1.set_zorder(ax2.get_zorder() + 1)
ax1.patch.set_visible(False)

line1, = ax1.plot(
    df_boston['business_date'],
    df_boston['occupancy'],
    label='Current Books (OTB)',
    color='#2b5c8f',
    linewidth=2.5
)

line2, = ax1.plot(
    df_boston['business_date'],
    df_boston['forecast_occupancy'],
    label='90-Day Forecast',
    color='#e67e22',
    linewidth=2.5,
    linestyle='--'
)

cap_line = ax1.axhline(1.0, color='#e74c3c', linestyle=':', linewidth=2, label='100% Capacity')

ax1.set_ylabel('Occupancy %', color='black', fontsize=12, fontweight='bold', labelpad=15)
ax1.set_ylim(0, 1.15)
ax1.yaxis.set_major_formatter(PercentFormatter(1.0))

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(fontsize=11)
ax1.tick_params(axis='x', rotation=45)

lines = [line1, line2, cap_line, events_bars]
labels = [l.get_label() for l in lines]
ax1.legend(
    lines, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.12),
    ncol=4,
    frameon=False,
    fontsize=11
)

plt.title('90-Day Forecast vs. Event Demand (Boston, MA)', fontsize=18, fontweight='bold', pad=40)
plt.tight_layout()
plt.show()

In [ ]:
df_santa_monica = df_forecast[df_forecast['location'] == 'Santa Monica, CA'].copy()

fig, ax1 = plt.subplots(figsize=(15, 7), facecolor='white')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()

events_bars = ax2.bar(
    df_santa_monica['business_date'],
    df_santa_monica['total_event_visitors'],
    alpha=0.15,
    color='#800080',
    width=1.2,
    label='Event Visitors'
)

ax2.set_ylabel('Total Event Visitors', color='#800080', fontsize=12, fontweight='bold', labelpad=15)
ax2.grid(False)

def thousands_formatter(x, pos):
    return f'{int(x/1000)}K' if x >= 1000 else str(int(x))
ax2.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))

ax1.set_zorder(ax2.get_zorder() + 1)
ax1.patch.set_visible(False)

line1, = ax1.plot(
    df_santa_monica['business_date'],
    df_santa_monica['occupancy'],
    label='Current Books (OTB)',
    color='#2b5c8f',
    linewidth=2.5
)

line2, = ax1.plot(
    df_santa_monica['business_date'],
    df_santa_monica['forecast_occupancy'],
    label='90-Day Forecast',
    color='#e67e22',
    linewidth=2.5,
    linestyle='--'
)

cap_line = ax1.axhline(1.0, color='#e74c3c', linestyle=':', linewidth=2, label='100% Capacity')

ax1.set_ylabel('Occupancy %', color='black', fontsize=12, fontweight='bold', labelpad=15)
ax1.set_ylim(0, 1.15)
ax1.yaxis.set_major_formatter(PercentFormatter(1.0))

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(fontsize=11)
ax1.tick_params(axis='x', rotation=45)

lines = [line1, line2, cap_line, events_bars]
labels = [l.get_label() for l in lines]
ax1.legend(
    lines, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.12),
    ncol=4,
    frameon=False,
    fontsize=11
)

plt.title('90-Day Forecast vs. Event Demand (Santa Monica, CA)', fontsize=18, fontweight='bold', pad=40)
plt.tight_layout()
plt.show()